# New York Housing Market Analysis
**LAB University of Applied Sciences — NumPy / Pandas Assignment**

Dataset: `NY-House-Dataset.csv` — 4,801 property listings across New York City

This notebook covers:
- Loading and filtering CSV data with **pandas**
- Computing **Min, Max**, and key statistics
- **Charts**: Price vs Longitude and Price vs Latitude
- **Interactive map** of NYC housing prices with Folium

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster

print('All libraries imported successfully!')

---
## 2. Load the Dataset

In [ ]:
df = pd.read_csv('NY-House-Dataset.csv')

print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'Column names: {df.columns.tolist()}')
df.head()

---
## 3. Explore the Raw Data

In [ ]:
# Data types and null counts
df.info()

In [ ]:
# Basic statistics on numeric columns
df[['PRICE', 'BEDS', 'BATH', 'PROPERTYSQFT', 'LATITUDE', 'LONGITUDE']].describe()

In [ ]:
# Property types in the dataset
print('Property types:')
print(df['TYPE'].value_counts())

---
## 4. Filter the Data (pandas)

The raw data contains extreme outliers (e.g. price = 2,147,483,647 which is the max int32 value — a data error). We clean these out.

In [ ]:
print(f'Rows before filtering: {len(df)}')

# Filter 1: Remove unrealistic prices
df = df[(df['PRICE'] >= 50_000) & (df['PRICE'] <= 10_000_000)]

# Filter 2: Remove properties with 0 sqft
df = df[df['PROPERTYSQFT'] > 0]

# Filter 3: Keep only valid NYC coordinates
df = df[
    (df['LATITUDE']  >= 40.4) & (df['LATITUDE']  <= 41.0) &
    (df['LONGITUDE'] >= -74.3) & (df['LONGITUDE'] <= -73.6)
]

# Filter 4: Remove rows with missing values
df = df.dropna(subset=['PRICE', 'LATITUDE', 'LONGITUDE', 'PROPERTYSQFT'])

print(f'Rows after filtering: {len(df)}')
df.head()

In [ ]:
# Add a Price per Square Foot column
df['PRICE_PER_SQFT'] = df['PRICE'] / df['PROPERTYSQFT']

# Remove unrealistic price/sqft values
df = df[(df['PRICE_PER_SQFT'] > 1) & (df['PRICE_PER_SQFT'] < 30_000)]

print(f'Final dataset: {len(df)} rows')
df[['PRICE', 'PROPERTYSQFT', 'PRICE_PER_SQFT', 'TYPE', 'SUBLOCALITY', 'LATITUDE', 'LONGITUDE']].head()

---
## 5. Min, Max, and Key Statistics

In [ ]:
print('===== PRICE STATISTICS =====')
print(f"Minimum price:       ${df['PRICE'].min():>15,.0f}")
print(f"Maximum price:       ${df['PRICE'].max():>15,.0f}")
print(f"Mean price:          ${df['PRICE'].mean():>15,.0f}")
print(f"Median price:        ${df['PRICE'].median():>15,.0f}")
print(f"Std deviation:       ${df['PRICE'].std():>15,.0f}")

print('\n===== PRICE PER SQ FT =====')
print(f"Minimum:             ${df['PRICE_PER_SQFT'].min():>15,.2f}")
print(f"Maximum:             ${df['PRICE_PER_SQFT'].max():>15,.2f}")
print(f"Mean:                ${df['PRICE_PER_SQFT'].mean():>15,.2f}")

print('\n===== LOCATION RANGE =====')
print(f"Latitude  - Min: {df['LATITUDE'].min():.4f},  Max: {df['LATITUDE'].max():.4f}")
print(f"Longitude - Min: {df['LONGITUDE'].min():.4f}, Max: {df['LONGITUDE'].max():.4f}")

In [ ]:
# Price per sqft bins
print('===== PRICE PER SQFT - BINS =====')
bins = pd.cut(df['PRICE_PER_SQFT'], bins=5)
for i, interval in enumerate(bins.value_counts().sort_index().index):
    print(f'Bin {i}: ${interval.left:,.2f} - ${interval.right:,.2f}')

In [ ]:
# Most and least expensive listings
print('TOP 5 MOST EXPENSIVE:')
print(df.nlargest(5, 'PRICE')[['TYPE', 'SUBLOCALITY', 'PRICE', 'PROPERTYSQFT', 'PRICE_PER_SQFT']].to_string(index=False))

print('\nTOP 5 CHEAPEST:')
print(df.nsmallest(5, 'PRICE')[['TYPE', 'SUBLOCALITY', 'PRICE', 'PROPERTYSQFT', 'PRICE_PER_SQFT']].to_string(index=False))

---
## 6. Charts: Price vs Longitude and Price vs Latitude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Longitude vs Price per Sqft
axes[0].scatter(df['LONGITUDE'], df['PRICE_PER_SQFT'], color='red', alpha=0.35, s=8)
axes[0].set_title('Longitude vs. Price per Sqft', fontsize=13)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Price per Sqft ($)')
axes[0].grid(True, linestyle='--', alpha=0.4)

# Chart 2: Latitude vs Price per Sqft
axes[1].scatter(df['LATITUDE'], df['PRICE_PER_SQFT'], color='blue', alpha=0.35, s=8)
axes[1].set_title('Latitude vs. Price per Sqft', fontsize=13)
axes[1].set_xlabel('Latitude')
axes[1].set_ylabel('Price per Sqft ($)')
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('price_lon_lat_charts.png', dpi=150)
plt.show()
print('Saved: price_lon_lat_charts.png')

In [ ]:
# Geographic scatter - Lat vs Lon colored by price per sqft
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    df['LONGITUDE'],
    df['LATITUDE'],
    c=df['PRICE_PER_SQFT'],
    cmap='viridis',
    alpha=0.5,
    s=12
)

plt.colorbar(scatter, ax=ax, label='Price per Sqft ($)')
ax.set_title('NYC Housing - Geographic Price Distribution', fontsize=14)
ax.set_xlabel('LONGITUDE')
ax.set_ylabel('LATITUDE')
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('price_geo_distribution.png', dpi=150)
plt.show()
print('Saved: price_geo_distribution.png')

---
## 7. Price Distribution by Property Type

In [ ]:
top_types = df['TYPE'].value_counts().head(5).index
df_types  = df[df['TYPE'].isin(top_types)]

fig, ax = plt.subplots(figsize=(12, 5))

for ptype in top_types:
    subset = df_types[df_types['TYPE'] == ptype]['PRICE']
    ax.hist(subset, bins=40, alpha=0.5, label=ptype)

ax.set_title('Price Distribution by Property Type', fontsize=13)
ax.set_xlabel('Price ($)')
ax.set_ylabel('Count')
ax.legend(fontsize=8)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('price_by_type.png', dpi=150)
plt.show()
print('Saved: price_by_type.png')

---
## 8. Average Price by Neighbourhood (SUBLOCALITY)

In [ ]:
avg_by_area = (
    df.groupby('SUBLOCALITY')['PRICE']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
avg_by_area.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 15 Most Expensive Neighbourhoods (Average Price)', fontsize=13)
ax.set_xlabel('Neighbourhood')
ax.set_ylabel('Average Price ($)')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('avg_price_by_neighbourhood.png', dpi=150)
plt.show()
print('Saved: avg_price_by_neighbourhood.png')

---
## 9. Interactive Map - Cheapest Apartments per Square Foot

The map is saved as an HTML file. Open **`nyc_housing_map.html`** in your browser to explore it interactively.

In [ ]:
# Take the 300 cheapest listings by price per sqft
cheapest = df.nsmallest(300, 'PRICE_PER_SQFT')

# Create map centered on NYC
nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=11)

# Add title
title_html = '<h3 style="text-align:center;font-family:Arial">Cheapest Apartments per Square Foot - NYC</h3>'
nyc_map.get_root().html.add_child(folium.Element(title_html))

# Add clustered markers
cluster = MarkerCluster().add_to(nyc_map)

for _, row in cheapest.iterrows():
    popup_html = f"""
    <b>Type:</b> {row['TYPE']}<br>
    <b>Price:</b> ${row['PRICE']:,.0f}<br>
    <b>Price/sqft:</b> ${row['PRICE_PER_SQFT']:,.2f}<br>
    <b>Beds/Bath:</b> {row['BEDS']} bd / {row['BATH']} ba<br>
    <b>SqFt:</b> {row['PROPERTYSQFT']:,}<br>
    <b>Area:</b> {row['SUBLOCALITY']}
    """
    folium.Marker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        popup=folium.Popup(popup_html, max_width=260),
        icon=folium.Icon(color='blue', icon='home', prefix='fa')
    ).add_to(cluster)

nyc_map.save('nyc_housing_map.html')
print('Map saved as nyc_housing_map.html - open it in your browser!')
nyc_map

---
## 10. Summary

| Step | Description |
|------|-------------|
| Load | Read `NY-House-Dataset.csv` with pandas (4,801 rows, 17 columns) |
| Filter | Removed price outliers, zero sqft, invalid coordinates |
| Statistics | Min, max, mean, median, std for price and price/sqft |
| Price bins | Divided price/sqft into 5 equal-width bins |
| Charts | Longitude vs Price, Latitude vs Price, geographic scatter, price by type, avg by neighbourhood |
| Map | Interactive Folium map of 300 cheapest listings with popups |